<a href="https://colab.research.google.com/github/TediBalint/AI-Jegyzetek/blob/master/Dropout%20%C3%A9s%20Gaussian%20Noise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dropout és Gaussian Noise

Ebben a notebookban a **dropout** és **Gaussian noise** regularizációs technikákat vizsgáljuk.

## Tartalomjegyzék

1. Dropout alapjai
2. Dropout variánsok
3. Gaussian noise
4. Training vs inference
5. Összehasonlítás

## 1. Dropout alapjai

### Mi a dropout?

| Fázis | Működés |
|-------|--------|
| Training | Véletlenszerű neuronok kikapcsolása |
| Inference | Minden neuron aktív, súlyok skálázva |

### Matematikai leírás

$$h_i^{(l)} = \frac{m_i}{1-p} \cdot f(W_i^{(l)} h^{(l-1)} + b_i^{(l)})$$

ahol:
- $m_i \sim \text{Bernoulli}(1-p)$
- $p$: dropout probability
- $\frac{1}{1-p}$: inverted dropout skálázás

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import make_classification, make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
torch.manual_seed(42)

# Dropout vizualizáció
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Teljes hálózat
def draw_network(ax, title, dropout_mask=None):
    layers = [4, 6, 6, 2]

    for l, n_neurons in enumerate(layers):
        for i in range(n_neurons):
            y = i - n_neurons/2 + 0.5

            if dropout_mask is not None and l in dropout_mask:
                active = dropout_mask[l][i]
                color = 'blue' if active else 'lightgray'
                alpha = 1.0 if active else 0.3
            else:
                color = 'blue'
                alpha = 1.0

            circle = plt.Circle((l, y), 0.2, color=color, alpha=alpha)
            ax.add_patch(circle)

            # Kapcsolatok
            if l < len(layers) - 1:
                for j in range(layers[l+1]):
                    y2 = j - layers[l+1]/2 + 0.5
                    if dropout_mask is not None:
                        if l in dropout_mask and not dropout_mask[l][i]:
                            continue
                        if (l+1) in dropout_mask and not dropout_mask[l+1][j]:
                            continue
                    ax.plot([l+0.2, l+0.8], [y, y2], 'gray', alpha=0.3, linewidth=0.5)

    ax.set_xlim(-0.5, len(layers)-0.5)
    ax.set_ylim(-4, 4)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title(title)

# Teljes hálózat
draw_network(axes[0], 'Teljes hálózat')

# Dropout p=0.3
np.random.seed(42)
dropout_mask_03 = {
    1: np.random.random(6) > 0.3,
    2: np.random.random(6) > 0.3
}
draw_network(axes[1], 'Dropout p=0.3', dropout_mask_03)

# Dropout p=0.5
np.random.seed(42)
dropout_mask_05 = {
    1: np.random.random(6) > 0.5,
    2: np.random.random(6) > 0.5
}
draw_network(axes[2], 'Dropout p=0.5', dropout_mask_05)

plt.tight_layout()
plt.show()

In [ ]:
# Dropout hatása tanítás során

X, y = make_classification(n_samples=500, n_features=20, n_informative=10,
                          n_redundant=5, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train_t = torch.FloatTensor(X_train)
y_train_t = torch.FloatTensor(y_train)
X_test_t = torch.FloatTensor(X_test)
y_test_t = torch.FloatTensor(y_test)

def create_model(dropout_rate=0.0):
    return nn.Sequential(
        nn.Linear(20, 128),
        nn.ReLU(),
        nn.Dropout(dropout_rate),
        nn.Linear(128, 64),
        nn.ReLU(),
        nn.Dropout(dropout_rate),
        nn.Linear(64, 1),
        nn.Sigmoid()
    )

def train_model(model, epochs=200):
    optimizer = optim.Adam(model.parameters(), lr=0.01)
    criterion = nn.BCELoss()

    train_losses, test_losses = [], []

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        outputs = model(X_train_t).squeeze()
        loss = criterion(outputs, y_train_t)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())

        model.eval()
        with torch.no_grad():
            test_out = model(X_test_t).squeeze()
            test_loss = criterion(test_out, y_test_t)
            test_losses.append(test_loss.item())

    return train_losses, test_losses

# Különböző dropout rate-ek
dropout_rates = [0.0, 0.2, 0.4, 0.6]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for ax, dr in zip(axes, dropout_rates):
    torch.manual_seed(42)
    model = create_model(dr)
    train_l, test_l = train_model(model)

    ax.plot(train_l, 'b-', label='Train')
    ax.plot(test_l, 'r-', label='Test')
    ax.set_title(f'Dropout = {dr}')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Dropout rate hatása', fontsize=14)
plt.tight_layout()
plt.show()

## 2. Dropout variánsok

### Típusok

| Variáns | Leírás | Használat |
|---------|--------|----------|
| Standard | Neuronok kikapcsolása | FC rétegek |
| Spatial | Feature map-ek kikapcsolása | CNN |
| Alpha | SELU-hoz optimalizált | Self-normalizing |
| DropConnect | Súlyok kikapcsolása | Alternatíva |

In [ ]:
# Dropout variánsok demonstrálása

# Standard Dropout
dropout_std = nn.Dropout(p=0.5)

# Dropout1d (spatial for 1D sequences)
dropout_1d = nn.Dropout1d(p=0.5)

# Dropout2d (spatial for 2D feature maps)
dropout_2d = nn.Dropout2d(p=0.5)

# Alpha Dropout (for SELU)
dropout_alpha = nn.AlphaDropout(p=0.5)

# Teszt input
torch.manual_seed(42)
x_1d = torch.randn(1, 10)  # [batch, features]
x_2d = torch.randn(1, 4, 8, 8)  # [batch, channels, H, W]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Standard dropout
for _ in range(5):
    dropped = dropout_std(x_1d)
    axes[0, 0].bar(range(10), dropped.squeeze().numpy(), alpha=0.3)
axes[0, 0].bar(range(10), x_1d.squeeze().numpy(), alpha=0.5, color='black', label='Eredeti')
axes[0, 0].set_title('Standard Dropout (p=0.5)')
axes[0, 0].set_xlabel('Feature index')
axes[0, 0].legend()

# Dropout2d - egész csatornák
torch.manual_seed(42)
x_2d_dropped = dropout_2d(x_2d)

axes[0, 1].imshow(x_2d[0].sum(dim=0).numpy(), cmap='viridis')
axes[0, 1].set_title('Eredeti (4 csatorna összes)')

axes[1, 0].imshow(x_2d_dropped[0].sum(dim=0).numpy(), cmap='viridis')
axes[1, 0].set_title('Dropout2d után (csatornák kikapcsolva)')

# Alpha dropout - SELU-val
torch.manual_seed(42)
x_alpha = torch.randn(1, 10)
selu = nn.SELU()

x_selu = selu(x_alpha)
x_selu_dropped = dropout_alpha(x_selu)

axes[1, 1].bar(range(10), x_selu.squeeze().numpy(), alpha=0.7, label='SELU output')
axes[1, 1].bar(range(10), x_selu_dropped.squeeze().numpy(), alpha=0.5, label='Alpha Dropout után')
axes[1, 1].set_title('Alpha Dropout (SELU-hoz)')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

## 3. Gaussian Noise

### Zaj injektálás

$$\tilde{h} = h + \epsilon, \quad \epsilon \sim \mathcal{N}(0, \sigma^2)$$

### Előnyök

| Tulajdonság | Leírás |
|-------------|--------|
| Regularizáció | Robusztusabb feature-ök |
| Augmentáció | Implicit adat bővítés |
| Dropout approx | Folytonos approximáció |

In [ ]:
class GaussianNoise(nn.Module):
    """
    Gaussian noise réteg.
    Csak training közben aktív.
    """
    def __init__(self, std=0.1):
        super().__init__()
        self.std = std

    def forward(self, x):
        if self.training:
            noise = torch.randn_like(x) * self.std
            return x + noise
        return x

# Gaussian noise hatása
def create_model_with_noise(noise_std=0.0):
    return nn.Sequential(
        nn.Linear(20, 64),
        nn.ReLU(),
        GaussianNoise(noise_std),
        nn.Linear(64, 32),
        nn.ReLU(),
        GaussianNoise(noise_std),
        nn.Linear(32, 1),
        nn.Sigmoid()
    )

noise_stds = [0.0, 0.1, 0.3, 0.5]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for ax, std in zip(axes, noise_stds):
    torch.manual_seed(42)
    model = create_model_with_noise(std)
    train_l, test_l = train_model(model)

    ax.plot(train_l, 'b-', label='Train')
    ax.plot(test_l, 'r-', label='Test')
    ax.set_title(f'Gaussian Noise σ={std}')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Gaussian Noise regularizáció', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Input vs Hidden layer noise

def create_model_input_noise(noise_std):
    return nn.Sequential(
        GaussianNoise(noise_std),  # Input noise
        nn.Linear(20, 64),
        nn.ReLU(),
        nn.Linear(64, 32),
        nn.ReLU(),
        nn.Linear(32, 1),
        nn.Sigmoid()
    )

def create_model_hidden_noise(noise_std):
    return nn.Sequential(
        nn.Linear(20, 64),
        nn.ReLU(),
        GaussianNoise(noise_std),  # Hidden noise
        nn.Linear(64, 32),
        nn.ReLU(),
        GaussianNoise(noise_std),  # Hidden noise
        nn.Linear(32, 1),
        nn.Sigmoid()
    )

noise_std = 0.2

torch.manual_seed(42)
model_input = create_model_input_noise(noise_std)
train_input, test_input = train_model(model_input)

torch.manual_seed(42)
model_hidden = create_model_hidden_noise(noise_std)
train_hidden, test_hidden = train_model(model_hidden)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(train_input, 'b-', label='Train')
axes[0].plot(test_input, 'r-', label='Test')
axes[0].set_title(f'Input Noise (σ={noise_std})')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(train_hidden, 'b-', label='Train')
axes[1].plot(test_hidden, 'r-', label='Test')
axes[1].set_title(f'Hidden Layer Noise (σ={noise_std})')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Training vs Inference

### Fontos különbség

| Mód | Dropout | Noise | BatchNorm |
|-----|---------|-------|----------|
| Training | Aktív | Aktív | Running stats update |
| Eval | Inaktív | Inaktív | Running stats használat |

In [ ]:
# model.train() vs model.eval() demonstráció

torch.manual_seed(42)
model = nn.Sequential(
    nn.Linear(10, 20),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(20, 5)
)

x = torch.randn(1, 10)

# Training mode
model.train()
outputs_train = []
for _ in range(10):
    outputs_train.append(model(x).detach().numpy())

# Eval mode
model.eval()
outputs_eval = []
for _ in range(10):
    outputs_eval.append(model(x).detach().numpy())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training - változó output
for i, out in enumerate(outputs_train):
    axes[0].plot(out.flatten(), 'o-', alpha=0.5, label=f'Run {i+1}')
axes[0].set_title('Training mode (változó output)')
axes[0].set_xlabel('Output neuron')
axes[0].set_ylabel('Érték')
axes[0].grid(True, alpha=0.3)

# Eval - konstans output
for i, out in enumerate(outputs_eval):
    axes[1].plot(out.flatten(), 'o-', alpha=0.5, label=f'Run {i+1}')
axes[1].set_title('Eval mode (konstans output)')
axes[1].set_xlabel('Output neuron')
axes[1].set_ylabel('Érték')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Variancia összehasonlítás
var_train = np.var(outputs_train, axis=0).mean()
var_eval = np.var(outputs_eval, axis=0).mean()
print(f"Training mode output variancia: {var_train:.6f}")
print(f"Eval mode output variancia: {var_eval:.6f}")

## 5. Összehasonlítás

### Dropout vs Gaussian Noise vs Weight Decay

In [ ]:
# Regularizációs módszerek összehasonlítása

# No regularization
torch.manual_seed(42)
model_none = nn.Sequential(
    nn.Linear(20, 128),
    nn.ReLU(),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Linear(64, 1),
    nn.Sigmoid()
)

# Dropout
torch.manual_seed(42)
model_dropout = nn.Sequential(
    nn.Linear(20, 128),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(64, 1),
    nn.Sigmoid()
)

# Gaussian Noise
torch.manual_seed(42)
model_noise = nn.Sequential(
    nn.Linear(20, 128),
    nn.ReLU(),
    GaussianNoise(0.2),
    nn.Linear(128, 64),
    nn.ReLU(),
    GaussianNoise(0.2),
    nn.Linear(64, 1),
    nn.Sigmoid()
)

def train_with_wd(model, weight_decay=0.0, epochs=200):
    optimizer = optim.Adam(model.parameters(), lr=0.01, weight_decay=weight_decay)
    criterion = nn.BCELoss()

    train_losses, test_losses = [], []

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        outputs = model(X_train_t).squeeze()
        loss = criterion(outputs, y_train_t)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())

        model.eval()
        with torch.no_grad():
            test_out = model(X_test_t).squeeze()
            test_loss = criterion(test_out, y_test_t)
            test_losses.append(test_loss.item())

    return train_losses, test_losses

# Train all
train_none, test_none = train_with_wd(model_none, 0.0)
train_dropout, test_dropout = train_with_wd(model_dropout, 0.0)
train_noise, test_noise = train_with_wd(model_noise, 0.0)

# Weight decay (új modell)
torch.manual_seed(42)
model_wd = nn.Sequential(
    nn.Linear(20, 128),
    nn.ReLU(),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Linear(64, 1),
    nn.Sigmoid()
)
train_wd, test_wd = train_with_wd(model_wd, weight_decay=0.01)

# Vizualizáció
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Train loss
axes[0].plot(train_none, label='No reg')
axes[0].plot(train_dropout, label='Dropout')
axes[0].plot(train_noise, label='Gaussian Noise')
axes[0].plot(train_wd, label='Weight Decay')
axes[0].set_title('Train Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Test loss
axes[1].plot(test_none, label='No reg')
axes[1].plot(test_dropout, label='Dropout')
axes[1].plot(test_noise, label='Gaussian Noise')
axes[1].plot(test_wd, label='Weight Decay')
axes[1].set_title('Test Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Összehasonlítás táblázat
print("\nVégső test loss:")
print(f"  No regularization: {test_none[-1]:.4f}")
print(f"  Dropout (0.3):     {test_dropout[-1]:.4f}")
print(f"  Gaussian (0.2):    {test_noise[-1]:.4f}")
print(f"  Weight Decay:      {test_wd[-1]:.4f}")

## Összefoglalás

### Dropout:

| Tulajdonság | Érték |
|-------------|-------|
| Típus | Bináris maszk |
| Tipikus p | 0.2-0.5 |
| Hol | FC rétegek után |

### Gaussian Noise:

| Tulajdonság | Érték |
|-------------|-------|
| Típus | Folytonos zaj |
| Tipikus σ | 0.1-0.3 |
| Hol | Input vagy hidden |

### Best practices:

1. `model.train()` tanításkor
2. `model.eval()` kiértékeléskor
3. Ne használj mindkettőt FC és BN együtt